# 🇻🇳 ViMind: Quy Trình Huấn Luyện Tự Động Từ Đầu (Pre-training & SFT)
Notebook này được tối ưu hóa cho Kaggle GPU (Tesla T4 / P100 16GB VRAM):
1. **Tự động clone mã nguồn từ GitHub.**
2. **Tự động tải dữ liệu tiếng Việt** (Wikipedia 405k bài viết & SFT 52k hội thoại).
3. **Tự động nhận diện Base Model** (từ Kaggle Dataset `vimind-pretrained-base` nếu có, bỏ qua 2h Pretrain).
4. **Tinh chỉnh chỉ dẫn (SFT):** Huấn luyện mô hình trở thành Trợ lý ảo đối thoại.

In [ ]:
# 1. Clone toàn bộ mã nguồn ViMind từ GitHub và di chuyển vào thư mục làm việc
!rm -rf /kaggle/working/vimind
!git clone https://github.com/WuKong0601/ViMind.git /kaggle/working/vimind
%cd /kaggle/working/vimind


In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt


In [ ]:
# 3. Kiểm tra thông số phần cứng GPU & Khả năng tương thích CUDA
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU Device: {gpu_name} (CUDA Capability {cap[0]}.{cap[1]})')


In [ ]:
# 4. Tự động tải dữ liệu SFT qua mạng cáp quang tốc độ cao
!python data_pipeline/download_sft.py


In [ ]:
# 5. Chạy bộ kiểm thử toàn diện (Pipeline Test & Dry Runs) đảm bảo 0 lỗi trước khi train
!python trainer/test_pipeline.py
!python trainer/test_dry_run.py
!python trainer/test_sft_dry_run.py


In [ ]:
# 6. [KIỂM TRA BASE MODEL & CHUẨN BỊ]
import os

base_model_path = None
candidates = [
    '/kaggle/input/vimind-pretrained-base',
    '/kaggle/input/vimind-pretrained-base/vimind_26m_final',
    'out/vimind_26m_final'
]
for p in candidates:
    if os.path.exists(p) and (os.path.exists(os.path.join(p, 'model.safetensors')) or os.path.exists(os.path.join(p, 'config.json'))):
        base_model_path = p
        break

if base_model_path:
    print(f'✅ Đã tìm thấy trọng số Base Model tại: {base_model_path}')
    print('⚡ Bỏ qua giai đoạn Pre-training (tiết kiệm ~2 giờ) và chuyển ngay sang SFT!')
else:
    print('Chưa có Base Model. Tiến hành tải Wikipedia và huấn luyện Pre-train từ đầu...')
    !python data_pipeline/download_pretrain.py
    !python trainer/pretrain.py \
        --data_path dataset/pretrain_vi.jsonl \
        --tokenizer_dir model \
        --save_dir out \
        --save_weight vimind_26m \
        --batch_size 32 \
        --accumulation_steps 4 \
        --epochs 1 \
        --dtype float16 \
        --log_interval 50 \
        --save_interval 1000
    base_model_path = 'out/vimind_26m_final'


In [ ]:
# 7. [GIAI ĐOẠN SFT] Tinh chỉnh chỉ dẫn từ trọng số Pre-training để biến thành Chatbot
import os
print(f'🚀 Bắt đầu SFT Fine-Tuning với trọng số nền: {base_model_path}...')
!python -u trainer/train_sft.py \
    --data_path dataset/sft_vi.jsonl \
    --tokenizer_dir model \
    --from_pretrained {base_model_path} \
    --save_dir out/sft \
    --save_weight vimind_sft \
    --batch_size 16 \
    --accumulation_steps 4 \
    --epochs 2 \
    --learning_rate 1e-4 \
    --dtype float16 \
    --log_interval 25 \
    --save_interval 500


In [ ]:
# 8. [KIỂM THỬ THÀNH PHẨM] Trò chuyện thử với mô hình ViMind vừa được huấn luyện xong
import os
import torch
from transformers import AutoTokenizer
from model.model import ViMindForCausalLM

model_path = 'out/sft/vimind_sft_final' if os.path.exists('out/sft/vimind_sft_final') else ('out/sft/vimind_sft_step_1000' if os.path.exists('out/sft/vimind_sft_step_1000') else base_model_path)
if os.path.exists(model_path):
    tokenizer = AutoTokenizer.from_pretrained('model')
    model = ViMindForCausalLM.from_pretrained(model_path).cuda()
    model.eval()

    test_prompts = [
        'Xin chào, bạn là ai và bạn có thể giúp gì cho tôi?',
        'Thủ đô của Việt Nam là gì?',
        'Hãy nêu 3 lợi ích của việc tập thể dục mỗi ngày.',
        'Làm thế nào để học lập trình Python hiệu quả?'
    ]

    print('=' * 60)
    print(f'🎉 KẾT QUẢ TRẢ LỜI CỦA VIMIND THÀNH PHẨM ({model_path}):')
    print('=' * 60)
    for p in test_prompts:
        formatted = tokenizer.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors='pt').input_ids.cuda()
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=150,
                temperature=0.7,
                top_p=0.85,
                top_k=20,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )
        ans = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        print(f'\n👤 Người dùng: {p}')
        print(f'🤖 ViMind: {ans.strip()}')
        print('-' * 60)
else:
    print(f'⚠️ Checkpoint not found at: {model_path}')
